### 📊 __Dataset Reference & Schema Definitions__
1. berlin_clubs_raw.csv
Description: A clean dimension table containing structural information about top-tier nightlife venues across primary Berlin operational districts.

Granularity: One row per unique club_id.

2. berlin_clubs_surge_windows.csv
Description: An expanded, programmatic lookup table that maps the exact closing hour and isolates the immediate 0 to 30-minute post-closing window for every active night.

Granularity: One row per club, per operating day.

### ⚙️ __Data Transformation Logic (The Code Mechanics)__
⏱️ Strict 30-Minute Boundary Constraint
The Logic: Explicitly labels the target block as 00-30_mins. Once a club closes, it gives 30 minutes as a buffer for last people to call for their taxi.

Why it matters: It reflects a tight, realistic passenger waiting window. Instead of generalized hourly analysis, this allows the downstream warehouse layer to filter for trips occurring strictly within the first half-hour of closing.

⚡ Feature Engineering: demand_weight
The Logic: Maps text labels (Large, Medium, Small) into linear numeric weights (3, 2, 1).

Why it matters: Databases cannot easily calculate statistical correlations or multiply coefficients against strings. Converting crowd sizes into numeric weights allows the SQL layer to instantly run math calculations against fleet revenue.

In [ ]:
import pandas as pd

DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

def generate_session(start_day, start_hour, end_day, end_hour):
    """
    Generates (day_of_week, hour, is_closing_rush) for every hour in a session,
    correctly rolling over the calendar day for overnight/multi-day sessions.
    """
    start_idx = DAYS.index(start_day)
    end_idx = DAYS.index(end_day)
    rows = []
    current_day_idx = start_idx
    current_hour = start_hour
    while True:
        is_last = (current_day_idx % 7 == end_idx and current_hour == end_hour)
        rows.append((DAYS[current_day_idx % 7], current_hour, is_last))
        if is_last:
            break
        current_hour += 1
        if current_hour == 24:
            current_hour = 0
            current_day_idx += 1
    return rows

# 1. Static Berlin Club Dataset — sessions defined with explicit start/end day+hour
clubs_data = [
    {
        'club_id': 'CLB_001', 'club_name': 'Berghain', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Large',
        'sessions': [
            # "Friday night" = Saturday 00:00, runs to 08:00
            {'start_day': 'Saturday', 'start_hour': 0, 'end_day': 'Saturday', 'end_hour': 8},
            # "Saturday night" = Sunday 00:00, runs to Monday 10:00 (the marathon)
            {'start_day': 'Sunday', 'start_hour': 0, 'end_day': 'Monday', 'end_hour': 10},
        ]
    },
    {
        'club_id': 'CLB_002', 'club_name': 'Tresor', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Large',
        'sessions': [
            {'start_day': 'Friday', 'start_hour': 2, 'end_day': 'Friday', 'end_hour': 12},
            {'start_day': 'Saturday', 'start_hour': 2, 'end_day': 'Saturday', 'end_hour': 12},
            {'start_day': 'Sunday', 'start_hour': 2, 'end_day': 'Sunday', 'end_hour': 12},
        ]
    },
    {
        'club_id': 'CLB_003', 'club_name': 'About Blank', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Medium',
        'sessions': [
            {'start_day': 'Friday', 'start_hour': 2, 'end_day': 'Friday', 'end_hour': 9},
            {'start_day': 'Saturday', 'start_hour': 2, 'end_day': 'Saturday', 'end_hour': 9},
        ]
    },
    {
        'club_id': 'CLB_004', 'club_name': 'Sisyphos', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Large',
        'sessions': [
            # Opens Friday 23:00, runs continuously until Monday 10:00
            {'start_day': 'Friday', 'start_hour': 23, 'end_day': 'Monday', 'end_hour': 10},
        ]
    },
    {
        'club_id': 'CLB_005', 'club_name': 'Ohm', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Medium',
        'sessions': [
            {'start_day': 'Friday', 'start_hour': 2, 'end_day': 'Friday', 'end_hour': 8},
            {'start_day': 'Saturday', 'start_hour': 2, 'end_day': 'Saturday', 'end_hour': 8},
            {'start_day': 'Sunday', 'start_hour': 2, 'end_day': 'Sunday', 'end_hour': 8},
        ]
    },
    {
        'club_id': 'CLB_006', 'club_name': 'Ritter Butzke', 'district': 'Friedrichshain-Kreuzberg', 'crowd_size': 'Medium',
        'sessions': [
            {'start_day': 'Friday', 'start_hour': 2, 'end_day': 'Friday', 'end_hour': 8},
            {'start_day': 'Saturday', 'start_hour': 2, 'end_day': 'Saturday', 'end_hour': 8},
        ]
    },
    {
        'club_id': 'CLB_008', 'club_name': 'KitKatClub', 'district': 'Mitte', 'crowd_size': 'Large',
        'sessions': [
            # Opens Friday 23:00, closes Saturday 07:00
            {'start_day': 'Friday', 'start_hour': 23, 'end_day': 'Saturday', 'end_hour': 7},
            # Opens again Saturday 20:00, closes Sunday 19:00 (7pm)
            {'start_day': 'Saturday', 'start_hour': 20, 'end_day': 'Sunday', 'end_hour': 19},
        ]
    },
    {
        'club_id': 'CLB_009', 'club_name': 'Golden Gate', 'district': 'Mitte', 'crowd_size': 'Medium',
        'sessions': [
            {'start_day': 'Friday', 'start_hour': 0, 'end_day': 'Friday', 'end_hour': 23},
            {'start_day': 'Saturday', 'start_hour': 0, 'end_day': 'Saturday', 'end_hour': 23},
            {'start_day': 'Sunday', 'start_hour': 0, 'end_day': 'Sunday', 'end_hour': 15},
        ]
    }
]

# 2. Flatten clubs into raw export
flat_clubs = []
for c in clubs_data:
    flat_clubs.append({
        'club_id': c['club_id'],
        'club_name': c['club_name'],
        'district': c['district'],
        'crowd_size': c['crowd_size']
    })
df_clubs = pd.DataFrame(flat_clubs)

# 3. Expand sessions into hourly surge windows (Filtering for 4 AM onwards)
surge_logs = []
for club in clubs_data:
    for session in club['sessions']:
        for day, hour, is_closing in generate_session(
            session['start_day'], session['start_hour'],
            session['end_day'], session['end_hour']
        ):
            # NEW LOGIC: Only count as an active taxi hail hour if it's 4:00 AM or later, 
            # OR if it's the final closing rush hour.
            is_peak_active = 0
            if not is_closing and hour >= 4:
                is_peak_active = 1
                
            # Keep the row if it's a peak active hour OR if it's the closing rush
            if is_peak_active == 1 or is_closing == 1:
                surge_logs.append({
                    'club_id': club['club_id'],
                    'club_name': club['club_name'],
                    'district': club['district'],
                    'day_of_week': day,
                    'surge_hour': hour,
                    'is_active_operating_hour': is_peak_active,
                    'is_closing_rush': 1 if is_closing else 0,
                    'demand_weight': 3 if club['crowd_size'] == 'Large' else 2 if club['crowd_size'] == 'Medium' else 1
                })

df_active = pd.DataFrame(surge_logs)

# 3.5 Clean up duplicates and prioritize closing rush indicators
df_active = df_active.sort_values(by=['club_id', 'day_of_week', 'surge_hour', 'is_closing_rush'], ascending=[True, True, True, False])
df_active = df_active.drop_duplicates(subset=['club_id', 'day_of_week', 'surge_hour'], keep='first')

# 4. Export
df_clubs.to_csv('berlin_clubs.csv', index=False)
df_active.to_csv('berlin_clubs_active_windows.csv', index=False)

print(f"Success! Saved {len(df_clubs)} unique clubs.")
print(f"Success! Generated {len(df_active)} granular operational hourly tracking windows.")